# Bronze layer

**What this notebook produces:** a working, idempotent bronze table, plus the numbers
that justify every decision in [`docs/bronze.md`](../docs/bronze.md).

Order of business:

1. Look at what the source actually gave us
2. Write down the schema contract
3. Build the schema check *(the one thing you write yourself)*
4. Add provenance and partition columns
5. Predict the output file count, then write, then check the prediction
6. **Append the same month twice and watch the count double.** Once. Then delete it
7. Rebuild correctly with dynamic partition overwrite and prove it is idempotent

Keep the Spark UI open at http://localhost:4040 the whole time.

## 0. Setup

In [1]:
import sys, shutil
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, str(Path.cwd().parent))   # so `import src.…` works from notebooks/

from pyspark.sql import functions as F
from src.session import get_spark
from src import config

spark = get_spark("bronze")
sc = spark.sparkContext

print("spark", spark.version, "| cores:", sc.defaultParallelism)
print("raw   :", config.RAW_YELLOW)
print("bronze:", config.BRONZE)
print("months found:", config.available_months())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/16 06:21:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.9 | AQE=False | UI http://wc-allauddin-shaik-shared-vpc:4040
spark 3.5.9 | cores: 2
raw   : /home/user/repo/sandbox/spark-nyc/data/yellow
bronze: /home/user/repo/sandbox/spark-nyc/data/bronze/yellow
months found: [(2024, 1), (2024, 2), (2024, 3)]


### Who actually runs this: driver and executor

⚠️ **"Master" and "worker" are Standalone-mode words.** The words that are true
everywhere are **driver** and **executor**.

**What you have right now (`local[*]`): one JVM. There is no master and no worker.**

```
  ┌──────────────────── ONE JVM, one machine ─────────────────────┐
  │                                                               │
  │   DRIVER                          EXECUTOR                    │
  │   your Python, over py4j          2 task slots = your 2 cores │
  │   ┌────────────────────┐          ┌─────────┐  ┌─────────┐    │
  │   │ builds the plan    │ tasks →  │ task 0  │  │ task 1  │    │
  │   │ decides partitions │          │ part-0  │  │ part-1  │    │
  │   │ schedules tasks    │ ← status │         │  │         │    │
  │   │ serves UI :4040    │          └─────────┘  └─────────┘    │
  │   └────────────────────┘                                      │
  └───────────────────────────────────────────────────────────────┘
```

**What it becomes on a cluster: separate machines, separate JVMs.**

```
   DRIVER  ─────────────►  CLUSTER MANAGER   (YARN / Kubernetes / Standalone)
   plans, schedules,               │          ^ THIS is what "master" means
   collects results                │            in Standalone mode
                                   ▼ allocates containers
              ┌────────────────┬────────────────┬────────────────┐
              │   EXECUTOR 1   │   EXECUTOR 2   │   EXECUTOR 3   │
              │   4 cores      │   4 cores      │   4 cores      │
              │   8 GB heap    │   8 GB heap    │   8 GB heap    │
              └────────────────┴────────────────┴────────────────┘
                     ▲                                    ▲
                     └───── a SHUFFLE moves data ─────────┘
                            between executors, over the network
```

| Word | What it is |
|---|---|
| **Driver** | Runs your code. Builds the plan, decides partition counts, schedules tasks. One per application |
| **Executor** | A JVM process that runs tasks and holds cached data. Many per application |
| **Core / slot** | One task at a time. An executor with 4 cores runs 4 tasks concurrently |
| **Task** | One partition being processed by one thread. The unit of work |
| **Worker node** | A machine that can host executors. Standalone-mode word |
| **Cluster manager** | Hands out resources. YARN, Kubernetes, or Spark Standalone (whose process is called the master) |

**Why this matters for your job:** it is a single stage with no shuffle, so nothing
ever crosses between executors. On a cluster it would run exactly the same way, just
with more slots. That is why local mode teaches the engine honestly.

## 1. What did the source give us?

Bronze's whole job is to absorb a boundary you do not control. So the first move is
always the same: **look at the thing before you touch it.**

In [2]:
path = config.raw_month_path(2024, 3)
df = spark.read.parquet(str(path))

print("input partitions:", df.rdd.getNumPartitions())
print("columns         :", len(df.columns))
print()
for f in df.schema.fields:
    print(f"  {f.name:<24} {f.dataType.simpleString()}")

input partitions: 2
columns         : 19

  VendorID                 int
  tpep_pickup_datetime     timestamp_ntz
  tpep_dropoff_datetime    timestamp_ntz
  passenger_count          bigint
  trip_distance            double
  RatecodeID               bigint
  store_and_fwd_flag       string
  PULocationID             int
  DOLocationID             int
  payment_type             bigint
  fare_amount              double
  extra                    double
  mta_tax                  double
  tip_amount               double
  tolls_amount             double
  improvement_surcharge    double
  total_amount             double
  congestion_surcharge     double
  Airport_fee              double


Neither of those read a single row of data. Parquet carries its schema in a **footer**
at the end of the file, so `printSchema` reads a few kilobytes. That is the difference
between Parquet and CSV in one sentence, and it is why the schema check below is cheap.

### Predict before you run the next cell

You counted March on day 1. Write the number down, then run it.

In [3]:
sc.setJobDescription("bronze | raw count 2024-03")
n_raw = df.count()
print(f"{n_raw:,}")

3,582,628


## 2. Write the contract down

"Expected schema" has to live **outside the data**, or "expected" just means "whatever
arrived", and a check that compares the file to itself always passes.

The cell below prints a dict you paste into the next cell. This is the only time it is
generated from data. From here on it is a constant that a human edits deliberately.

In [4]:
print("EXPECTED = {")
for f in df.schema.fields:
    print(f'    "{f.name}": "{f.dataType.simpleString()}",')
print("}")

EXPECTED = {
    "VendorID": "int",
    "tpep_pickup_datetime": "timestamp_ntz",
    "tpep_dropoff_datetime": "timestamp_ntz",
    "passenger_count": "bigint",
    "trip_distance": "double",
    "RatecodeID": "bigint",
    "store_and_fwd_flag": "string",
    "PULocationID": "int",
    "DOLocationID": "int",
    "payment_type": "bigint",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "Airport_fee": "double",
}


## 3. The schema check ⭐ you write this one

This is the piece of logic worth twenty minutes, because it is the whole reason bronze
exists as a layer. Everything else in this notebook is plumbing.

**The rules, from `docs/bronze.md` §5:**

| Change at the source | Do |
|---|---|
| New column appears | **Accept**, print it loudly |
| Expected column missing | **Raise** |
| Type changed | **Raise** |
| Column order changed | Ignore. We never read by position |

The three set expressions you need are already written for you. What is left is
deciding what to do with them, and writing an error message that a person woken at 3am
can act on. `ValueError("schema mismatch")` is a useless error. Name the columns and
name the types.

In [5]:
EXPECTED = {
    "VendorID": "int",
    "tpep_pickup_datetime": "timestamp_ntz",
    "tpep_dropoff_datetime": "timestamp_ntz",
    "passenger_count": "bigint",
    "trip_distance": "double",
    "RatecodeID": "bigint",
    "store_and_fwd_flag": "string",
    "PULocationID": "int",
    "DOLocationID": "int",
    "payment_type": "bigint",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "Airport_fee": "double",
}


def check_schema(df, expected=EXPECTED):
    '''Fail loudly if an incoming file does not match the contract.'''
    actual = {f.name: f.dataType.simpleString() for f in df.schema.fields}

    missing = set(expected) - set(actual)
    added   = set(actual) - set(expected)
    retyped = {c for c in set(expected) & set(actual) if expected[c] != actual[c]}

    if added:
        print(f"Warning: added columns detected: {added}")
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if retyped:
        raise ValueError(f"Columns with incorrect types: {retyped}")

**The test.** Run this once you think it works. Four cases, three of them should not pass silently.

In [6]:
def expect(label, fn, should_raise):
    try:
        fn()
        print(f"{'FAIL' if should_raise else 'ok  '}  {label}")
    except Exception as e:
        print(f"{'ok  ' if should_raise else 'FAIL'}  {label}  ->  {type(e).__name__}: {e}")

expect("unchanged        ", lambda: check_schema(df), should_raise=False)
expect("column added     ", lambda: check_schema(df.withColumn("surge_fee", F.lit(1.0))), should_raise=False)
expect("column missing   ", lambda: check_schema(df.drop("passenger_count")), should_raise=True)
expect("column retyped   ", lambda: check_schema(df.withColumn("trip_distance", F.col("trip_distance").cast("string"))), should_raise=True)

ok    unchanged        
ok    column added     
ok    column missing     ->  ValueError: Missing columns: {'passenger_count'}
ok    column retyped     ->  ValueError: Columns with incorrect types: {'trip_distance'}


## 4. Provenance and partition columns

Five added columns. Three say where the row came from, two say where it goes on disk.

Note what `year` and `month` are: **literals from the filename**, not anything derived
from `tpep_pickup_datetime`. The March file contains trips dated 2009. Deciding those
are wrong is a judgement, and judgements live in silver.

In [7]:
year, month = 2024, 3
batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

bronze_df = (
    df
    .withColumn(config.COL_SOURCE_FILE, F.input_file_name())
    .withColumn(config.COL_INGESTED_AT, F.current_timestamp())
    .withColumn(config.COL_BATCH_ID,    F.lit(batch_id))
    .withColumn("year",  F.lit(year))
    .withColumn("month", F.lit(month))
)

print("batch:", batch_id)
(bronze_df
   .select(config.COL_SOURCE_FILE, config.COL_INGESTED_AT, config.COL_BATCH_ID, "year", "month")
   .show(3, truncate=False))

batch: 20260816T064243Z
+------------------------------------------------------------------------------------+--------------------------+----------------+----+-----+
|_source_file                                                                        |_ingested_at              |_batch_id       |year|month|
+------------------------------------------------------------------------------------+--------------------------+----------------+----+-----+
|file:///home/user/repo/sandbox/spark-nyc/data/yellow/yellow_tripdata_2024-03.parquet|2026-08-16 06:42:43.830948|20260816T064243Z|2024|3    |
|file:///home/user/repo/sandbox/spark-nyc/data/yellow/yellow_tripdata_2024-03.parquet|2026-08-16 06:42:43.830948|20260816T064243Z|2024|3    |
|file:///home/user/repo/sandbox/spark-nyc/data/yellow/yellow_tripdata_2024-03.parquet|2026-08-16 06:42:43.830948|20260816T064243Z|2024|3    |
+------------------------------------------------------------------------------------+--------------------------+-----------

Check the Spark UI. **Five `withColumn` calls produced zero jobs.** `show(3)` produced
one. That is lazy evaluation doing its job, and it is why you never optimise by
counting lines of transformation code.

### Predict the output file count

No shuffle happens in this write, so the number of output files per partition equals
the number of Spark partitions holding that data. You have the formula from day 2:

```
totalBytes    = fileSize + openCostInBytes
bytesPerCore  = totalBytes / defaultParallelism
maxSplitBytes = min(maxPartitionBytes, max(openCostInBytes, bytesPerCore))
```

March is 57.3 MB, `openCostInBytes` is 4 MiB, and you know your core count. Work out
how many files you expect in `year=2024/month=3/`, write it down, then run the write.

## 5. Write it

In [8]:
out = str(config.BRONZE)

sc.setJobDescription("bronze | write 2024-03")
(bronze_df.write
    .mode("append")
    .partitionBy("year", "month")
    .parquet(out))

for p in sorted(Path(out).rglob("*")):
    if p.is_file() and not p.name.startswith("_"):
        print(f"{p.stat().st_size/1e6:8.1f} MB   {p.relative_to(out)}")

     0.0 MB   ._SUCCESS.crc
     0.3 MB   year=2024/month=3/.part-00000-a2d3359c-b99f-41a1-9542-1a3d7078f364.c000.snappy.parquet.crc
     0.2 MB   year=2024/month=3/.part-00001-a2d3359c-b99f-41a1-9542-1a3d7078f364.c000.snappy.parquet.crc
    43.2 MB   year=2024/month=3/part-00000-a2d3359c-b99f-41a1-9542-1a3d7078f364.c000.snappy.parquet
    30.2 MB   year=2024/month=3/part-00001-a2d3359c-b99f-41a1-9542-1a3d7078f364.c000.snappy.parquet


Look at the paths. `year=2024/month=3/` is **one partition**, a folder. The `part-*`
files inside it are just files, one per writing task. Those are two different levels and
conflating them is the most common Spark misreading there is.

Also note `month=3`, not `month=03`. Add a tenth month and directory listings sort as
1, 10, 11, 12, 2, 3. It works, it just reads badly. Decide now whether you care.

In [9]:
back = spark.read.parquet(out)
print(f"rows      {back.count():,}   (source was {n_raw:,})")
print(f"columns   {len(back.columns)}   = 19 source + 3 provenance + 2 partition")
back.select("year", "month").distinct().show()

rows      3,582,628   (source was 3,582,628)
columns   24   = 19 source + 3 provenance + 2 partition


+----+-----+
|year|month|
+----+-----+
|2024|    3|
+----+-----+



**`year` and `month` came back, but they are not in the Parquet files.** Spark stripped
them out when writing, because the directory path already encodes the value, and
reconstructed them from the path on read. That is why a partition column costs no
storage no matter how long its name is.

---

## 6. The experiment: run it twice

This is the whole reason approach A is wrong. Do it once, see the number, delete it.

In [10]:
sc.setJobDescription("bronze | write 2024-03 AGAIN (append)")
(bronze_df.write
    .mode("append")
    .partitionBy("year", "month")
    .parquet(out))

print(f"after second append: {spark.read.parquet(out).count():,}   (should be {n_raw:,})")

after second append: 7,165,256   (should be 3,582,628)


No error. No warning. Every downstream number is now inflated by exactly 100% and
nothing in the system knows.

That is the failure mode this layer exists to prevent, and it is why "the job succeeded"
is not the same claim as "the data is right".

Clear it and build it properly.

In [11]:
shutil.rmtree(out, ignore_errors=True)
print("cleared:", out)

cleared: /home/user/repo/sandbox/spark-nyc/data/bronze/yellow


## 7. The correct version

One config line changes the meaning of `mode("overwrite")`:

| `partitionOverwriteMode` | What `overwrite` does |
|---|---|
| `static` **(the default)** | Deletes **the entire table**, writes only the incoming partitions |
| `dynamic` | Replaces only the partitions present in the incoming data |

The default is the most destructive default in Spark. Re-running one month with
`static` silently drops the other eleven.

In [12]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


def land_month(year: int, month: int, batch_id: str) -> None:
    src = spark.read.parquet(str(config.raw_month_path(year, month)))
    check_schema(src)

    (src
        .withColumn(config.COL_SOURCE_FILE, F.input_file_name())
        .withColumn(config.COL_INGESTED_AT, F.current_timestamp())
        .withColumn(config.COL_BATCH_ID,    F.lit(batch_id))
        .withColumn("year",  F.lit(year))
        .withColumn("month", F.lit(month))
        .write
        .mode("overwrite")
        .partitionBy("year", "month")
        .parquet(str(config.BRONZE)))


def run_all() -> str:
    batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    for y, m in config.available_months():
        sc.setJobDescription(f"bronze | {y}-{m:02d} | batch {batch_id}")
        land_month(y, m, batch_id)
    return batch_id

In [13]:
b1 = run_all()
print(f"run 1  batch={b1}  rows={spark.read.parquet(str(config.BRONZE)).count():,}")

b2 = run_all()
print(f"run 2  batch={b2}  rows={spark.read.parquet(str(config.BRONZE)).count():,}")

run 1  batch=20260816T070816Z  rows=9,554,778


run 2  batch=20260816T070907Z  rows=9,554,778


**Same number twice. That is idempotency**, and it is a property of the *result*, not of
the run. A job that succeeds twice and doubles the rows also succeeded twice.

What it is still not: **atomic**. Kill the kernel halfway through that write and you get
a half-replaced partition, because the delete and the write are separate steps. That is
the gap Iceberg closes, and it is the honest reason to reach for it here rather than
"Iceberg is the modern way".

In [14]:
(spark.read.parquet(str(config.BRONZE))
   .groupBy("year", "month", config.COL_BATCH_ID)
   .count()
   .orderBy("year", "month")
   .show(truncate=False))

+----+-----+----------------+-------+
|year|month|_batch_id       |count  |
+----+-----+----------------+-------+
|2024|1    |20260816T070907Z|2964624|
|2024|2    |20260816T070907Z|3007526|
|2024|3    |20260816T070907Z|3582628|
+----+-----+----------------+-------+



Only the second batch id survives. That is the proof the write **replaced** rather than
appended, and it is the observability property earning its keep: you can tell which run
produced every row in the table.

---

## 8. Fill this in before moving on

| Question | Predicted | Actual |
|---|---|---|
| Rows in March | | |
| Files in `year=2024/month=3/` | | |
| Bronze size on disk vs the 160 MB source | | |
| Rows after the double append | | |
| Rows after two correct runs | | |

Any row where predicted and actual disagree is the only interesting row on the page.
Put the reason in `docs/bronze.md` §10.

## 9. What goes into `src/bronze/ingest.py`

Four things, lifted straight out of this notebook:

- `EXPECTED` and `check_schema()` → their own module, `src/bronze/schema.py`
- `land_month(year, month, batch_id)` → unchanged
- `run_all()` → renamed `main()`, so Airflow can call it as one task
- the `partitionOverwriteMode` config → moves into `src/session.py`, so no caller can
  forget it and trigger the static-overwrite footgun

Then `python -m src.bronze.ingest` twice, and the row count does not move.

---

## 10. Controlling how many files come out

You got 2 files per month because the read formula decided 2 splits, and **a write
with no shuffle produces one file per Spark partition**. Three levers, at different
points in the pipeline:

| Lever | When it applies | Shuffle? |
|---|---|---|
| `spark.sql.files.maxPartitionBytes` | Read time. Changes how the input is split | No |
| `.coalesce(n)` | Before write. **Only reduces.** Can leave uneven partitions | No |
| `.repartition(n)` | Before write. Any n, evenly distributed | **Yes, full shuffle** |

Rule of thumb: **`coalesce` to reduce, `repartition` to redistribute.** `coalesce(1)`
on a big job is a classic way to make everything run on one core.

In [ ]:
out = str(config.BRONZE)

def file_count(month):
    d = Path(out) / "year=2024" / f"month={month}"
    return len([f for f in d.glob("*.parquet")])

# --- lever 1: read side. Smaller splits, no shuffle. ---
spark.conf.set("spark.sql.files.maxPartitionBytes", 16 * 1024 * 1024)   # 16 MB
print("16MB splits  ->", spark.read.parquet(str(config.raw_month_path(2024, 3))).rdd.getNumPartitions(), "partitions")
spark.conf.set("spark.sql.files.maxPartitionBytes", 512 * 1024 * 1024)  # 512 MB
print("512MB splits ->", spark.read.parquet(str(config.raw_month_path(2024, 3))).rdd.getNumPartitions(), "partitions")
spark.conf.set("spark.sql.files.maxPartitionBytes", 128 * 1024 * 1024)  # back to default

# --- lever 2: write side. ONE file per month. ---
b = build(2024, 3, "filecount-test") if "build" in dir() else bronze_df
b.repartition(1).write.mode("overwrite").partitionBy("year", "month").parquet(out)
print("after repartition(1) ->", file_count(3), "file(s)")

# --- lever 3: write side. FOUR files. ---
b.repartition(4).write.mode("overwrite").partitionBy("year", "month").parquet(out)
print("after repartition(4) ->", file_count(4 - 1), "file(s)")

---

# 11. Iceberg

> ⚠️ **Restart the kernel before this section.** `spark.jars.packages` is read when the
> JVM launches, and the JVM survives `spark.stop()`. A running kernel can never pick up
> the Iceberg jars. Restart, then run the cell below and skip everything above it.

What changes: **one line of write code.** The schema check, the provenance columns and
`run_all` are all untouched. What you get back is everything you had to do by hand.

| What you did with plain Parquet | With Iceberg |
|---|---|
| `shutil.rmtree` to recover from a bad run | `CALL system.rollback_to_snapshot(...)` |
| No way to see which batch wrote which file | `SELECT * FROM …files` |
| `_SUCCESS` vanished, nothing signals completion | The commit **is** the signal |
| A crash mid-write leaves a half-replaced partition | Atomic. Invisible until the pointer moves |

In [ ]:
import sys, json
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, str(Path.cwd().parent))
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from src.session import get_spark
from src import config

spark = get_spark("bronze-iceberg", iceberg=True)   # first run downloads ~30 MB
sc = spark.sparkContext
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")
print("catalog ready")

In [ ]:
def build(year: int, month: int, batch_id: str):
    return (spark.read.parquet(str(config.raw_month_path(year, month)))
        .withColumn(config.COL_SOURCE_FILE, F.input_file_name())
        .withColumn(config.COL_INGESTED_AT, F.current_timestamp())
        .withColumn(config.COL_BATCH_ID,    F.lit(batch_id))
        .withColumn("year",  F.lit(year))
        .withColumn("month", F.lit(month)))


# Create the table EMPTY, so the schema is registered without landing any data.
y0, m0 = config.available_months()[0]
(build(y0, m0, "init").limit(0)
    .writeTo("local.bronze.yellow")
    .partitionedBy(col("year"), col("month"))
    .createOrReplace())

spark.sql("DESCRIBE TABLE local.bronze.yellow").show(30, truncate=False)

In [ ]:
def run_all_iceberg() -> str:
    batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    for y, m in config.available_months():
        sc.setJobDescription(f"iceberg | {y}-{m:02d} | batch {batch_id}")
        build(y, m, batch_id).writeTo("local.bronze.yellow").overwritePartitions()
    return batch_id

b1 = run_all_iceberg()
print(f"run 1  batch={b1}  rows={spark.table('local.bronze.yellow').count():,}")
b2 = run_all_iceberg()
print(f"run 2  batch={b2}  rows={spark.table('local.bronze.yellow').count():,}")

In [ ]:
print("=== SNAPSHOTS: every write, in order ===")
spark.sql("""
  SELECT snapshot_id, parent_id, operation,
         summary['added-records']   AS added,
         summary['deleted-records'] AS deleted
  FROM local.bronze.yellow.snapshots ORDER BY committed_at
""").show(truncate=False)

print("=== FILES: this IS the manifest, as a table ===")
spark.sql("""
  SELECT partition, record_count, file_size_in_bytes,
         substring_index(file_path, '/', -1) AS file
  FROM local.bronze.yellow.files ORDER BY partition
""").show(truncate=False)

print("=== MANIFESTS ===")
spark.sql("SELECT added_data_files_count, existing_data_files_count, deleted_data_files_count, partition_summaries FROM local.bronze.yellow.manifests").show(truncate=False)

In [ ]:
snaps = [r[0] for r in spark.sql(
    "SELECT snapshot_id FROM local.bronze.yellow.snapshots ORDER BY committed_at").collect()]
print("snapshots:", snaps)

# Time travel: read the table as it was, without changing anything
for s in snaps:
    n = spark.sql(f"SELECT count(*) c FROM local.bronze.yellow VERSION AS OF {s}").first().c
    print(f"  snapshot {s} -> {n:,} rows")

In [ ]:
# Roll the whole table back to an earlier snapshot. This is the replacement for rmtree.
target = snaps[len(snaps) // 2]
spark.sql(f"CALL local.system.rollback_to_snapshot('bronze.yellow', {target})")
print(f"rolled back to {target}: {spark.table('local.bronze.yellow').count():,} rows")

# And forward again, because a rollback is itself just another commit
spark.sql(f"CALL local.system.set_current_snapshot('bronze.yellow', {snaps[-1]})")
print(f"forward to  {snaps[-1]}: {spark.table('local.bronze.yellow').count():,} rows")

In [ ]:
meta = Path("../warehouse/bronze/yellow/metadata")
for f in sorted(meta.iterdir()):
    print(f"{f.stat().st_size:>8,}  {f.name}")

print("\n--- version-hint.text: THE CATALOG POINTER, made visible ---")
print(repr((meta / "version-hint.text").read_text()))

latest = sorted(meta.glob("v*.metadata.json"), key=lambda p: int(p.stem[1:].split('.')[0]))[-1]
d = json.loads(latest.read_text())
print(f"\n--- {latest.name} ---")
print("format-version      :", d["format-version"])
print("current-snapshot-id :", d["current-snapshot-id"])
print("schemas             :", len(d["schemas"]))
print("partition specs     :", [s["fields"] for s in d["partition-specs"]])
print("\nsnapshots:")
for s in d["snapshots"]:
    print(f"  {s['snapshot-id']}  {s['operation']:<9}  ->  {s['manifest-list'].split('/')[-1]}")

### What to actually look at

1. **`version-hint.text` holds a single number.** That is the entire catalog pointer for
   a hadoop catalog. Committing means writing a new number into that file. Now you can
   see exactly why it cannot do a safe compare-and-swap, and why production needs a REST
   catalog or a database.
2. **The snapshot count grows but the row count does not.** Old data files are still on
   disk, referenced only by old snapshots. That is what time travel costs.
3. **`operation` says `overwrite`, and `deleted-records` is non-zero on run 2.** The
   partition really was replaced, not appended to.
4. **`…files` gives you `partition`, `record_count` and file size per file.** Directory
   listing is never needed again.

### Predictions, filled in

| Question | Predicted | Actual |
|---|---|---|
| Rows in March | | |
| Files in `year=2024/month=3/` | | |
| Bronze size vs the 153 MB source | | |
| Rows after the double append | | |
| Snapshots after two Iceberg runs | | |

Put anything that surprised you into `docs/bronze.md` §10.